# 5G‑NIDD — Augmentation Preparation (Conditional + Unconditional) — CTGAN

This notebook prepares the 5G‑NIDD flow dataset for **tabular augmentation** and **downstream classifier training**.

**Outputs** (written to `OUT_DIR`)
- `train_input.csv`, `test_input.csv`
- `train_input_with_label.csv`, `test_input_with_label.csv`
- `prep_metadata.json` (prep trace)
- `prep_metadata_aug.json` (augmentation-ready, used by training notebooks)
- `prep_preparation_summary.json` (timing + counts + checks)


## 0) Configuration

In [ ]:
INPUT_CSV  = "Datasets/train_per_class/Encoded_train_UDPScan_10000.csv"       
OUT_DIR    = "5G-NIDD_Aug_CTGAN/UDPScan"  

# Label to use for conditional augmentation
LABEL_COL  = "Attack Type"

# Run mode: "unconditional" | "conditional" | "both"
MODE       = "both"

# Test set 
TEST_CSV   = "Datasets/test_per_class/Encoded_test_UDPScan_3181.csv"
RANDOM_SEED = 42

# Columns to keep (leave as None to keep *all*)
KEEP_COLS = [
    "Dur","SrcPkts","DstPkts","SrcBytes","DstBytes","sMeanPktSz",
    "Load","SrcLoad","DstLoad","Loss","Attack Type",
    "arp","icmp","ipv6-icmp","llc","lldp","sctp","tcp","udp",
    "ACC","CON","ECO","FIN","INT","NRS","REQ","RSP","RST","TST","URP",
    "Shutdown","Start","Status","39","4","52","54",
    "af11","af12","af41","cs0","cs4","cs6","cs7","ef"
]

# Columns to drop.
DROP_COLS = [
    "sTos","dTos","sTtl","dTtl","sHops","dHops",
    "TcpRtt","SynAck","AckDat","Offset","sVid","dVid",
    "SrcTCPBase","DstTCPBase","SrcWin","DstWin",
    "pLoss","SrcLoss","DstLoss","SrcGap","DstGap",
    "Rate","SrcRate","DstRate"
]

# Loss handling: "auto" (binarize if ultra-rare), "keep", or "drop"
LOSS_MODE = "auto"

# Is this a sub-dataset --> Lable dataset like ICMPFlood, etc.
IS_SUBDATASET = True # Set to True if working on a sub-dataset enables dropping constant columns based on big dataset.
GLOBAL_CONSTANT_ONEHOTS = ['ipv6-icmp', 'NRS', '39', '54']  # Constant one-hot columns from the big dataset to drop in sub-dataset.

# Heavy-tail stabilization (Bytes/Load-like features)
USE_LOG1P = True
LOG1P_COLS = ["SrcBytes","DstBytes","Load","SrcLoad","DstLoad"]

# Scaling: "standard" or "robust"
SCALER = "standard"

import os, json, time, platform, numpy as np, pandas as pd
from pathlib import Path
from collections import Counter

np.random.seed(RANDOM_SEED)
os.makedirs(OUT_DIR, exist_ok=True)

print("INPUT_CSV:", INPUT_CSV)
print("OUT_DIR:", OUT_DIR)
print("LABEL_COL:", LABEL_COL)
print("MODE:", MODE)


## 1) Load dataset

In [ ]:
import pandas as pd

# Robust CSV loader (skips malformed lines if any)
def read_csv_robust(path: str) -> pd.DataFrame:
    return pd.read_csv(path, engine="python", on_bad_lines="skip")

t0 = time.time()
df_train = read_csv_robust(INPUT_CSV)
df_test  = read_csv_robust(TEST_CSV)
load_seconds = time.time() - t0

print("Train Rows, Cols:", df_train.shape)
print("Test  Rows, Cols:", df_test.shape)
print("Load time (s):", round(load_seconds, 2))
print("First 20 columns:", df_train.columns[:20].tolist())
display(df_train.head(5))


## 2) Detect one‑hot columns

In [ ]:
# Detect binary one-hot columns (strict {0,1} after numeric coercion)
def is_binary_onehot(s: pd.Series) -> bool:
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return False
    u = set(s.unique().tolist())
    return u.issubset({0,1,0.0,1.0})


## 3) Drop constants and loss handling

In [ ]:
# 1) Drop obvious index columns if present (both train + test)
for _df_name in ["df_train", "df_test"]:
    _df = globals()[_df_name]
    idx_cols = [c for c in ["Unnamed: 0", "index"] if c in _df.columns]
    if idx_cols:
        globals()[_df_name] = _df.drop(columns=idx_cols)
        print(f"Dropped index-like columns in {_df_name}: {idx_cols}")

# 2) Apply KEEP_COLS / DROP_COLS (both)
if KEEP_COLS is not None:
    missing_train = [c for c in KEEP_COLS if c not in df_train.columns]
    missing_test  = [c for c in KEEP_COLS if c not in df_test.columns]
    assert not missing_train, f"KEEP_COLS missing in train: {missing_train}"
    assert not missing_test,  f"KEEP_COLS missing in test: {missing_test}"
    df_train = df_train.loc[:, KEEP_COLS].copy()
    df_test  = df_test.loc[:,  KEEP_COLS].copy()
    print("Applied KEEP_COLS. Train shape:", df_train.shape, "Test shape:", df_test.shape)

if DROP_COLS:
    drop_train = [c for c in DROP_COLS if c in df_train.columns]
    drop_test  = [c for c in DROP_COLS if c in df_test.columns]
    if drop_train:
        df_train = df_train.drop(columns=drop_train)
    if drop_test:
        df_test = df_test.drop(columns=drop_test)
    if drop_train or drop_test:
        print("Dropped DROP_COLS. Train shape:", df_train.shape, "Test shape:", df_test.shape)

# 3) Ensure label presence for conditional/both (both)
if MODE in ("conditional", "both"):
    assert LABEL_COL in df_train.columns, f"Label column '{LABEL_COL}' not found in TRAIN."
    assert LABEL_COL in df_test.columns,  f"Label column '{LABEL_COL}' not found in TEST."

# 4) log1p transform for heavy-tailed numeric features (deterministic; apply to both)
if USE_LOG1P:
    for _c in LOG1P_COLS:
        if _c != LABEL_COL:
            if _c in df_train.columns:
                x = pd.to_numeric(df_train[_c], errors="coerce").fillna(0)
                df_train[_c] = np.log1p(np.clip(x, 0, None))
            if _c in df_test.columns:
                x = pd.to_numeric(df_test[_c], errors="coerce").fillna(0)
                df_test[_c] = np.log1p(np.clip(x, 0, None))

# Normalize label text for stable conditional augmentation (both)
if MODE in ("conditional", "both") and LABEL_COL in df_train.columns:
    for _df_name in ["df_train", "df_test"]:
        _df = globals()[_df_name]
        _df[LABEL_COL] = _df[LABEL_COL].astype(str).str.strip()
        _df[LABEL_COL] = _df[LABEL_COL].replace({"nan": "Unknown", "None": "Unknown"})
        _df[LABEL_COL] = _df[LABEL_COL].astype(str)

# 5) Handle Loss column consistently (train decides; applied to both)
loss_handling = "absent"
if "Loss" in df_train.columns:
    if LOSS_MODE == "drop":
        df_train = df_train.drop(columns=["Loss"])
        df_test  = df_test.drop(columns=["Loss"], errors="ignore")
        loss_handling = "dropped"
    elif LOSS_MODE == "keep":
        loss_handling = "kept_numeric"
    else:
        frac = (pd.to_numeric(df_train["Loss"], errors="coerce").fillna(0) > 0).mean()
        print("Loss non-zero fraction (train):", round(float(frac), 6))
        if frac < 0.01:
            df_train["has_loss"] = (pd.to_numeric(df_train["Loss"], errors="coerce").fillna(0) > 0).astype(int)
            df_test["has_loss"]  = (pd.to_numeric(df_test["Loss"], errors="coerce").fillna(0) > 0).astype(int)
            df_train = df_train.drop(columns=["Loss"])
            df_test  = df_test.drop(columns=["Loss"], errors="ignore")
            loss_handling = "binarized_to_has_loss"
        else:
            loss_handling = "kept_numeric"

print("Loss handling:", loss_handling)

# 6) Drop constant one-hot columns based on TRAIN only, then apply to BOTH

#Drop globally-known constant one-hot columns ONLY when working on a subdataset
if IS_SUBDATASET and GLOBAL_CONSTANT_ONEHOTS:
    drop_cols = [c for c in GLOBAL_CONSTANT_ONEHOTS if c in df_train.columns]

    if drop_cols:
        df_train = df_train.drop(columns=drop_cols)
        df_test  = df_test.drop(columns=drop_cols, errors="ignore")

    print(
        f"Dropped GLOBAL constant one-hots (subdataset={IS_SUBDATASET}):",
        drop_cols,
        "Train shape:", df_train.shape,
        "Test shape:", df_test.shape
    )
else:
    print("Skipped dropping global constant one-hots (IS_SUBDATASET=False)")

    onehot_candidates = [
        c for c in df_train.columns
        if c != LABEL_COL and pd.api.types.is_numeric_dtype(df_train[c]) and is_binary_onehot(df_train[c])
    ]
    onehot_constant = [c for c in onehot_candidates if df_train[c].nunique(dropna=True) <= 1]

    print("One-hot candidates (train):", len(onehot_candidates))
    print("Constant one-hots (drop in both):", len(onehot_constant))
    print("Example constant one-hots:", onehot_constant[:20])

    if onehot_constant:
        df_train = df_train.drop(columns=onehot_constant)
        df_test  = df_test.drop(columns=onehot_constant, errors="ignore")
        print("Dropped constant one-hots. Train shape:", df_train.shape, "Test shape:", df_test.shape)

# 7) Align column order (train as reference)
df_test = df_test.reindex(columns=df_train.columns, fill_value=np.nan)
print("Aligned columns. Train cols:", df_train.shape[1], "Test cols:", df_test.shape[1])

display(df_train.head(3))


## 4) Impute + scale numeric, keep one‑hots 0/1

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler

# Identify binary vs numeric columns (based on TRAIN and exclude label)
bin_cols, num_cols = [], []
for c in df_train.columns:
    if c == LABEL_COL:
        continue
    if pd.api.types.is_numeric_dtype(df_train[c]) and is_binary_onehot(df_train[c]):
        bin_cols.append(c)
    else:
        num_cols.append(c)

imp = SimpleImputer(strategy="median")
sca = RobustScaler() if SCALER == "robust" else StandardScaler()

t0 = time.time()
if num_cols:
    # Fit on TRAIN
    df_train[num_cols] = imp.fit_transform(df_train[num_cols])
    df_train[num_cols] = sca.fit_transform(df_train[num_cols])
    # Transform TEST with the same fitted objects
    df_test[num_cols]  = imp.transform(df_test[num_cols])
    df_test[num_cols]  = sca.transform(df_test[num_cols])
scale_seconds = time.time() - t0

print("Binary cols:", len(bin_cols))
print("Numeric/scaled cols:", len(num_cols))
print("Scale+impute time (s):", round(scale_seconds, 2))
display(df_train.head(3))


## 5) Write matrices

In [ ]:
# Build X/y for train and test
Xtr = df_train.drop(columns=[LABEL_COL], errors="ignore")
Xte = df_test.drop(columns=[LABEL_COL], errors="ignore")

if MODE in ("conditional", "both"):
    ytr = df_train[LABEL_COL].astype(str)
    yte = df_test[LABEL_COL].astype(str)
else:
    ytr = None
    yte = None

# Save unconditional inputs
train_input_path = os.path.join(OUT_DIR, "train_input.csv")
test_input_path  = os.path.join(OUT_DIR, "test_input.csv")
Xtr.to_csv(train_input_path, index=False)
Xte.to_csv(test_input_path, index=False)

cond_train_path = None
cond_test_path  = None
if MODE in ("conditional", "both"):
    tr_l = Xtr.copy()
    te_l = Xte.copy()
    tr_l[LABEL_COL] = ytr.values
    te_l[LABEL_COL] = yte.values
    cond_train_path = os.path.join(OUT_DIR, "train_input_with_label.csv")
    cond_test_path  = os.path.join(OUT_DIR, "test_input_with_label.csv")
    tr_l.to_csv(cond_train_path, index=False)
    te_l.to_csv(cond_test_path, index=False)

print("Wrote:")
print(" -", train_input_path, Xtr.shape)
print(" -", test_input_path,  Xte.shape)
if cond_train_path:
    print(" -", cond_train_path, tr_l.shape)
    print(" -", cond_test_path,  te_l.shape)

if MODE in ("conditional", "both"):
    test_counts = pd.Series(yte).value_counts()
    print("Min test count per class:", int(test_counts.min()))


## 6) Write prep metadata

In [ ]:
prep_meta = {
    "input_csv": INPUT_CSV,
    "test_csv": TEST_CSV,
    "out_dir": OUT_DIR,
    "mode": MODE,
    "label_col": LABEL_COL if MODE in ("conditional", "both") else None,
    "keep_cols": KEEP_COLS,
    "drop_cols": DROP_COLS,
    "loss_handling": loss_handling,
    "dropped_constant_onehots": onehot_constant if not IS_SUBDATASET else GLOBAL_CONSTANT_ONEHOTS,
    "binary_onehots_used": sorted(bin_cols),
    "numeric_scaled_columns": sorted([c for c in num_cols if c in Xtr.columns]),
    "train_shape": [int(Xtr.shape[0]), int(Xtr.shape[1])],
    "test_shape": [int(Xte.shape[0]), int(Xte.shape[1])],
    "final_column_order": list(df_train.columns),
    "timing_seconds": {"load": round(load_seconds, 2), "scale_impute": round(scale_seconds, 2)},
}

if MODE in ("conditional", "both"):
    label_values = sorted(df_train[LABEL_COL].dropna().astype(str).unique().tolist())
    label_counts = dict(Counter(df_train[LABEL_COL].astype(str)))
    prep_meta["label_values"] = label_values
    prep_meta["label_counts"] = label_counts
    prep_meta["num_classes"] = int(len(label_values))

prep_meta_path = os.path.join(OUT_DIR, "prep_metadata.json")
with open(prep_meta_path, "w") as f:
    json.dump(prep_meta, f, indent=2)
print("Wrote:", prep_meta_path)
prep_meta


## 7) Write augmentation metadata 
### This metadata is used by training notebooks

In [ ]:
def infer_onehots(df_: pd.DataFrame, exclude=None):
    exclude = set(exclude or [])
    cols = []
    for c in df_.columns:
        if c in exclude:
            continue
        s = pd.to_numeric(df_[c], errors="coerce").dropna()
        if len(s) == 0:
            continue
        u = set(s.unique().tolist())
        if u.issubset({0, 1, 0.0, 1.0}):
            cols.append(c)
    return sorted(cols)

#  Load prepared train files 
train_uncond_path = os.path.join(OUT_DIR, "train_input.csv")
Xtr_uncond = pd.read_csv(train_uncond_path, engine="python", on_bad_lines="skip")

if "bin_cols" in globals() and isinstance(bin_cols, (list, tuple)) and len(bin_cols) > 0:
    discrete_uncond = sorted({c for c in bin_cols if c in Xtr_uncond.columns})
else:
    discrete_uncond = infer_onehots(Xtr_uncond)

kept_columns_final = list(Xtr_uncond.columns)

aug_meta = {
    "source_metadata": os.path.basename(prep_meta_path) if "prep_meta_path" in globals() else None,
    "output_dir": OUT_DIR,
    "input_csv_original": prep_meta.get("input_csv") if "prep_meta" in globals() else None,
    "test_csv_original": prep_meta.get("test_csv") if "prep_meta" in globals() else None,

    "loss_handling": prep_meta.get("loss_handling") if "prep_meta" in globals() else None,
    "kept_columns_final": kept_columns_final,

    "train_stats": {
        "unconditional": {"rows": int(Xtr_uncond.shape[0]), "cols": int(Xtr_uncond.shape[1])}
    },
    "numeric_scaled_columns": prep_meta.get("numeric_scaled_columns", []) if "prep_meta" in globals() else [],

    "augmentation_unconditional": {
        "train_path": os.path.basename(train_uncond_path),
        "discrete_columns": discrete_uncond,
        "label_col": None
    },
}

# Conditional
if MODE in ("conditional", "both"):
    train_cond_path = os.path.join(OUT_DIR, "train_input_with_label.csv")
    Xtr_cond = pd.read_csv(train_cond_path, engine="python", on_bad_lines="skip")

    if "bin_cols" in globals() and isinstance(bin_cols, (list, tuple)) and len(bin_cols) > 0:
        discrete_cond_feats = sorted({c for c in bin_cols if c in Xtr_cond.columns and c != LABEL_COL})
    else:
        discrete_cond_feats = infer_onehots(Xtr_cond, exclude=[LABEL_COL])

    aug_meta["train_stats"]["conditional"] = {"rows": int(Xtr_cond.shape[0]), "cols": int(Xtr_cond.shape[1])}
    aug_meta["augmentation_conditional"] = {
        "train_path": os.path.basename(train_cond_path),
        "discrete_columns": sorted(set(discrete_cond_feats + [LABEL_COL])),
        "label_col": LABEL_COL,
        "label_values": prep_meta.get("label_values", []) if "prep_meta" in globals() else [],
        "label_counts": prep_meta.get("label_counts", {}) if "prep_meta" in globals() else {},
    }

out_path = os.path.join(OUT_DIR, "prep_metadata_aug.json")
with open(out_path, "w") as f:
    json.dump(aug_meta, f, indent=2)

print("Wrote:", out_path)

## 8) Validation

In [ ]:
def is_binary_onehot_strict(s: pd.Series) -> bool:
    s = pd.to_numeric(s, errors="coerce").dropna()
    if len(s) == 0:
        return False
    u = set(s.unique().tolist())
    return u.issubset({0, 1, 0.0, 1.0})

def validate_dataset(_df: pd.DataFrame, name: str):
    # Validate one-hots
    bad_bin = [c for c in bin_cols if c in _df.columns and not is_binary_onehot_strict(_df[c])]
    assert not bad_bin, f"[{name}] Non-binary values in one-hots: {bad_bin[:10]}"

    # Validate numeric scaled columns: no NaNs
    if num_cols:
        nan_rate = float(pd.DataFrame(_df[num_cols]).isna().mean().mean())
        assert nan_rate == 0.0, f"[{name}] NaNs remain in numeric columns (avg nan rate={nan_rate})"

    # Conditional label checks
    if MODE in ("conditional", "both"):
        assert LABEL_COL in _df.columns, f"[{name}] LABEL_COL missing"
        assert _df[LABEL_COL].nunique() >= 1, f"[{name}] Need >=1 label values"

    print(f"Validation OK ({name})")

validate_dataset(df_train, "TRAIN")
validate_dataset(df_test, "TEST")

## 9) Preparation Summary 

In [ ]:
import hashlib

def sha256_file(path, block_size=2**20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(block_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

summary = {
    "stage": "preparation",
    "mode": MODE,
    "input_csv": INPUT_CSV,
    "test_csv": TEST_CSV,
    "out_dir": OUT_DIR,
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "hostname": platform.node(),
    "files": {
        "train_input.csv": {"sha256": sha256_file(os.path.join(OUT_DIR, "train_input.csv"))},
        "test_input.csv":  {"sha256": sha256_file(os.path.join(OUT_DIR, "test_input.csv"))},
        "prep_metadata.json": {"sha256": sha256_file(os.path.join(OUT_DIR, "prep_metadata.json"))},
    },
}

if MODE in ("conditional", "both"):
    summary["files"]["train_input_with_label.csv"] = {"sha256": sha256_file(os.path.join(OUT_DIR, "train_input_with_label.csv"))}
    summary["files"]["test_input_with_label.csv"]  = {"sha256": sha256_file(os.path.join(OUT_DIR, "test_input_with_label.csv"))}

out_json = os.path.join(OUT_DIR, "prep_preparation_summary.json")
with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("Wrote:", out_json)
summary
